### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
#pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [4]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [5]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [6]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [7]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [8]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [9]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [10]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [11]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [12]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [13]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [14]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [15]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [16]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], dtype=int64)

Obtenemos los 5 documentos más similares:

In [17]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [18]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [19]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [20]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [21]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [22]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### 1)

In [25]:
import random

In [33]:
# How many docs:
len(newsgroups_train.data), X_train.shape

(11314, (11314, 101631))

In [40]:
# Let's pick 5 random documents.
doc_ids = sorted(random.sample(range(len(newsgroups_train.data)), 5))
doc_ids

[383, 5803, 8431, 9691, 10738]

In [48]:
# Similarity
cos_sim = [(idx, 
            np.flip(np.sort(cosine_similarity(X_train[idx], X_train)[0]))[1:6],
            np.flip(np.argsort(cosine_similarity(X_train[idx], X_train)[0]))[1:6],) 
           for idx in doc_ids]

In [49]:
cos_sim

[(383,
  array([0.2678516 , 0.22925275, 0.2233804 , 0.20601562, 0.20216838]),
  array([ 641, 6635, 6894, 3596, 4253], dtype=int64)),
 (5803,
  array([0.71682237, 0.65078478, 0.59395376, 0.58491838, 0.56121545]),
  array([8550, 8660, 2189, 3652, 3808], dtype=int64)),
 (8431,
  array([0.25043309, 0.21027536, 0.13295735, 0.12821904, 0.11988008]),
  array([ 1049,  9161,  9577,  9623, 11098], dtype=int64)),
 (9691,
  array([0.4560742 , 0.26607755, 0.24978455, 0.19842949, 0.15423166]),
  array([7821, 9172, 4119, 2919, 6218], dtype=int64)),
 (10738,
  array([0.33460669, 0.24418769, 0.18843146, 0.18570174, 0.17004641]),
  array([ 4660,   764, 10839,  2457,  3977], dtype=int64))]

#### 1st Doc

In [64]:
didx = cos_sim[0][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')

---
Well, a student body president can't exactly campaign on the stand
that he's "tough on crime".  Their job is to listen to what people want
and fund things that make sense.

Condoms and marijuana aren't exactly the worst things to have available
either...
---

Category: talk.politics.misc


We see the doc with index 383 talks about what's the job of a student body president, remarking their job isn't to enforce, but rather to follow the will of the students.

Category: Talk - Politics - Miscellaneous

In [65]:
didx = cos_sim[0][2][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[0][1][0]}')

--- 
You think that you all have it bad....here at good ol' Southwest Missouri
State U., we have 2 parties running for student body president.  There's the
token sorority/fraternity faces, and then there's the president and vice
president of NORML.  They campaigned by handing out condoms and listing
their qualifications as,"I listen really well."  It makes me sick to have
a party established on many of the things that are ruining this country like
they are.  I think I'll run next year.:(
---

Category: talk.politics.misc
Cosine Similarity: 0.2678515982364508


This document talks about the difficulties the writer faced when running for student body president. Makes sense there's some similarity between this and the main document, but since they only share the overall topic of "student body president" but talk about different aspects related to this concept, a cosine similarity of ~1/4 is reasonable. They also share the same class, which shows there's some similarity here.

In [66]:
didx = cos_sim[0][2][1]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[0][1][1]}')

---THE WHITE HOUSE

                    Office of the Press Secretary
______________________________________________________________
For Immediate Release                             April 15, 1993     

	     
                       REMARKS BY THE PRESIDENT
                   TO LAW ENFORCEMENT ORGANIZATIONS
	     
	     
                           The Rose Garden 


2:52 P.M. EDT


	     THE PRESIDENT:  Good afternoon.  Ladies and gentlemen, 
two months ago I presented a comprehensive plan to reduce our 
national deficit and to increase our investment in the American 
people, their jobs and their economic future.  The federal budget 
plan passed Congress in record time, and created a new sense of hope 
and opportunity in the country.  
	     
	     Then, the short-term jobs plan I presented to Congress, 
which would create a half a million jobs in the next two years passed 
the House of Representatives two weeks ago.  It now has the support 
of a majority of the United States Senate.

This seems to be a presidential address followed by a Q&A. I suppose it makes sense this document would share some similarity towards our main doc, since it mentions how the student body president should listen to the students and fund things that "make sense", and in this presidential address, it's mentioned a few times the president's intention to tackle issues affecting the american people. Maybe that's the reason for the similarity, because otherwise these two documents have very little in common, other than both being about a "president". Should be noted that they do share the same class.

In [67]:
didx = cos_sim[0][2][2]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[0][1][2]}')

---Here is a press release from the White House.

 President Clinton's Remarks On Waco With Q/A
 To: National Desk
 Contact: White House Office of the Press Secretary, 202-456-2100

   WASHINGTON, April 20 -- Following are remarks by President 
Clinton in a question and answer session with the press:

1:36 P.M. EDT

     THE PRESIDENT:  On February the 28th, four federal
agents were killed in the line of duty trying to enforce the law
against the Branch Davidian compound, which had illegally stockpiled
weaponry and ammunition, and placed innocent children at risk.
Because the BATF operation had failed to meet its objective, a 51-day
standoff ensued.

     The Federal Bureau of Investigation then made every
reasonable effort to bring this perilous situation to an end without
bloodshed and further loss of life.  The Bureau's efforts were
ultimately unavailing because the individual with whom they were
dealing, David Koresh, was dangerous, irrational, and probably
insane.

     He engaged

Same logic here. This a press release following an event involving violence. It doesn't share much similarity to our main document other than the figure of the president. They don't share a class in this case, since this document is classified under politics-guns, rather than politics-misc.

In [68]:
didx = cos_sim[0][2][3]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[0][1][3]}')

---THE WHITE HOUSE

                    Office of the Press Secretary
_________________________________________________________________
For Immediate Release                             April 14, 1993     

	     
                       REMARKS BY THE PRESIDENT
                      AT SUMMER JOBS CONFERENCE

	     	  
                            Hyatt Regency
                        Crystal City, Virginia  


11:22 A.M. EDT

	     
	     THE PRESIDENT:  Thank you very much.  The speech that 
Octavius gave says more than anything I will be able to say today 
about why it's important to give all of our young people a chance to 
get a work experience and to continue to learn, to merge the nature 
of learning and work; why it's important to honor the efforts of 
people like Jerry Levin and Nancye Combs and Pat Irving and all of 
those who are here.  
	     
	     I want to thank the Secretaries of Labor and Education 
and all the people who work with them for sponsoring this; and my 
good

Again, same situation. This is again a presidential address, this time for job program expansion. Not exactly similar to our main document other than the figure of the president. In this case, both documents share a class though.

In [69]:
didx = cos_sim[0][2][4]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[0][1][4]}')

---THE WHITE HOUSE

                  Office of the Press Secretary
                    (Pittsburgh, Pennsylvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

	     
                    INTERVIEW OF THE PRESIDENT
                      BY MICHAEL WHITELY OF
                    KDKA-AM RADIO, PITTSBURGH
	     
                 Pittsburgh International Airport
                     Pittsburgh, Pennsylvania    



10:40 A.M. EDT
	     
	     
	     Q	  For everyone listening on KDKA Radio, I'm Mike 
Whitely, KDKA Radio News.  We're here at the Pittsburgh 
International Airport and with me is the President of the United 
States Bill Clinton.
	     
	     And I'd like to welcome you to the area and to KDKA.
	     
	     THE PRESIDENT:  Thank you, Mike.  Glad to be here.
	     
	     Q	  There are a lot of things we'd like to talk 
about in the brief amount of time we have, but some news is just 
breaking fro

This is the same situation as the others. This is an interview of the president in which he talks about the outcome of a trial. Other than the concept of president, there's not much similarity. Same class as our main document.

#### 2nd Doc

In [70]:
didx = cos_sim[1][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')

---

Shouldn't have.  But he may need to see the shrink about why he
wanted to kill himself.  Depressed people can be succesfully treated
usually.





-- 
----------------------------------------------------------------------------
Gordon Banks  N3JXP      | "Skepticism is the chastity of the intellect, and
geb@cadre.dsl.pitt.edu   |  it is shameful to surrender it too soon." 
---

Category: sci.med


This document is an e-mail, apparently, where someone talks about how depressed people can usually be trated and recommends seeking a shrink, which comes from a third party's desire to end their own life at some point.

In [71]:
didx = cos_sim[1][2][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[1][1][0]}')

---

So just what was it you wanted to say?



-- 
----------------------------------------------------------------------------
Gordon Banks  N3JXP      | "Skepticism is the chastity of the intellect, and
geb@cadre.dsl.pitt.edu   |  it is shameful to surrender it too soon." 
---

Category: sci.med
Cosine Similarity: 0.7168223692638934


We observe a high similarity. This makes sense since it seems this document belongs not only to the same sender, but also to the same e-mail chain that the main document as pulled from. Could also be a discussion on some sort of forum? Where end of the document is the user's signature.

In [73]:
didx = cos_sim[1][2][1]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[1][1][1]}')

---

By law, they would not be allowed to do that anyhow.




-- 
----------------------------------------------------------------------------
Gordon Banks  N3JXP      | "Skepticism is the chastity of the intellect, and
geb@cadre.dsl.pitt.edu   |  it is shameful to surrender it too soon." 
---

Category: sci.med
Cosine Similarity: 0.6507847775407407


Same case here. HIgh similarity, product of it being from the same sender. The signature is a big part of the overall document, so the similarity with our main document is high.

In [74]:
didx = cos_sim[1][2][2]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[1][1][2]}')

---
Senile keratoses.  Have nothing to do with the liver.


-- 
----------------------------------------------------------------------------
Gordon Banks  N3JXP      | "Skepticism is the chastity of the intellect, and
geb@cadre.dsl.pitt.edu   |  it is shameful to surrender it too soon." 
---

Category: sci.med
Cosine Similarity: 0.5939537603429336


Same here. Further discussion is not warranted, in my opinion.

In [75]:
didx = cos_sim[1][2][3]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[1][1][3]}')

---

"Diet Evangelist".  Good term.  Fits Atkins to a "T".  


-- 
----------------------------------------------------------------------------
Gordon Banks  N3JXP      | "Skepticism is the chastity of the intellect, and
geb@cadre.dsl.pitt.edu   |  it is shameful to surrender it too soon." 
---

Category: sci.med
Cosine Similarity: 0.5849183812053883


Same here. Further discussion is not warranted, in my opinion.

In [76]:
didx = cos_sim[1][2][4]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[1][1][4]}')

---
There is eye dominance same as handedness (and usually for the
same side).  It has nothing to do with refractive error, however.


-- 
----------------------------------------------------------------------------
Gordon Banks  N3JXP      | "Skepticism is the chastity of the intellect, and
geb@cadre.dsl.pitt.edu   |  it is shameful to surrender it too soon." 
---

Category: sci.med
Cosine Similarity: 0.5612154477387862


Same here. Further discussion is not warranted, in my opinion.

#### 3rd Doc

In [77]:
didx = cos_sim[2][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')

---From article <1qvgu5INN2np@lynx.unm.edu>, by osinski@chtm.eece.unm.edu (Marek Osinski):


Thessaloniki is called Thessaloniki by its inhabitants for the last 2300 years.
The city was never called Solun by its inhabitants.
Instabul was called Konstantinoupolis from 320 AD until about the 1920s.
That's about 1600 years. There many people alive today who were born in a city
called Konstantinoupolis. How many people do you know that were born in a city 
called Solun.
---

Category: talk.politics.mideast


This document seems to talk about how inhabitants of cities refer to said cities. In particular, how Thessaloniki was never referred to as Solun by the people that live there.

In [78]:
didx = cos_sim[2][2][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[2][1][0]}')

---

Well, it did not take long to see how consequent some Greeks are in
requesting that Thessaloniki are not called Solun by Bulgarian netters. 
So, Napoleon, why do you write about Konstantinople and not Istanbul?
---

Category: talk.politics.mideast
Cosine Similarity: 0.25043309293390237


This one has around 1/4 similarity to the main document. It mentions the Thessaloniki Solun issue, and it could very well be a fragment of a dicussion in a forum from which the main doc was extracted, but it's hard to know for sure. They do share the same class.

In [79]:
didx = cos_sim[2][2][1]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[2][1][1]}')

---Osinski):
years.
city

Are you one of those people who were born when Istanbul was called 
Konstantinopolis? I don't think so! If those people use it because
they are used to do so, then I understand. But open any map
today (except a few that try to be political) you will see that the name 
of the city is printed as Istanbul. So, don't try to give
any arguments to using Konstantinopolis except to cause some
flames, to make some political statement. 


--
Tankut Atan
tankut@iastate.edu
---

Category: talk.politics.mideast
Cosine Similarity: 0.21027536108222472


With around 1/5 similarity, this document seems to tackle and argument made by the main document, which used Istanbul as an example. It makes sense to see this document among the most similar, since it seems to be a fragment of the same discussion.

In [81]:
didx = cos_sim[2][2][2]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[2][1][2]}')

---If the  new  Kuiper belt object *is*  called 'Karla', the next
one  should be called 'Smiley'.
---

Category: sci.space
Cosine Similarity: 0.13295735288230248


This one seems to be completely unrelated to the main document, since it talks about the Kuiper belt. The category is also science-space, rather than talk-politics-middleeast. If I were to throw a guess as to why it appears among the most similar, it's cause it also discusses "naming" of things, in this case astronomical objects in the Kuiper belt. The main document discusses the names of cities.

In [82]:
didx = cos_sim[2][2][3]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[2][1][3]}')

---Accounts of Anti-Armenian Human Right Violations in Azerbaijan #012
                 Prelude to Current Events in Nagorno-Karabakh

        +---------------------------------------------------------+
        |                                                         |
        |  I saw a naked girl with her hair down. They were       |
        |  dragging her. She kept falling because they were       |
        |  pushing her and kicking her. She fell down, it was     |
        |  muddy there, and later other witnesses who saw it from |
        |  their balconies told us, they seized her by the hair   |
        |  and dragged her a couple of blocks, as far as the      |
        |  mortgage bank, that's a good block and a half or two   |
        |  from here. I know this for sure because I saw it       |
        |  myself.                                                |
        |                                                         |
        +----------------------------------------

While it shares category with our main document, this one has nothing to do with it. It's a testimony which describes planned right violations towards armenian people.

In [83]:
didx = cos_sim[2][2][4]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[2][1][4]}')

---
Let's see, if Alexander destroyed Tyre, and people move back, and
they construct houses, and after a while 14000 people live there
and still call it Tyre, it is not considered to be rebuilt. Instead
it's considered to be 'just-some-people-that-got-together-for-fishing-
and-they-needed-houses' place.


Sigh, I was never born in a city then (my home town has 10.000
people). I have to consult my city and inform them that it's from
now a fishing village. When this city (Kristinestad) was founded
in the 17:th century about 1000 people lived there, so the norms
were even more bizarre for dumb Swedish queens who founded cities
along the coast of Finland.

I would like to know why Paul thought is was worth mentioning the 
small fishing place of Tyre in Acts. Again, maybe he was a keen
fisherman and wanted to visit the shores of Tyre? :-)

Cheers,
Kent
---

Category: talk.religion.misc
Cosine Similarity: 0.11988007759211539


Same here. This document doesn't even share category with our main document, and bears no connection to the main document, other than mentioning a city and giving an example of the city of Tyre and it's inhabitants calling it Tyre. But the issue seems to be completely different.

#### 4th Doc

In [84]:
didx = cos_sim[3][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')

---
Actually, Messier was invited, but declined due to nagging injuries...
Keenan and Messier have always gotten along...Keenan dumped Steve
Yzerman from the last Canada Cup team, even though Yzerman had
endured the training camp, when Messier who had missed essentially the
entire camp recovering from injuries became available at the
last moment.
---

Category: rec.sport.hockey


This document talks about a hockey player declining the join the national team due to injury, and how another player was removed even when completing the training camp.

In [85]:
didx = cos_sim[3][2][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[3][1][0]}')

---
Messier was not invited due to his nagging injuries.  While the press
made an issue of it, and attempted to link it to the Rangers' internal
political woes, Mike Keenan repeated that to Messier personally during
the MSG press conference.  It makes sense ... Messier would probably
have not declined the invitation if it were made for publicity ...

gld
---

Category: rec.sport.hockey
Cosine Similarity: 0.45607420082259


We see around 1/2 similarity. Both documents talk about the same overall issue, a player not being on the national team due to injury, but they provide conflicting accounts. Main doc claims he was invited, while this one claims he wasn't.

In [86]:
didx = cos_sim[3][2][1]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[3][1][1]}')

---
But ultimately their hockey philosophies are like night and day...
Keenan believes in pressuring the opposition and taking the
initiative (within the limits of his system)...while Roger
has a reactive hockey philosophy...which is why Messier will
be able to and has played for Keenan, but thought Roger's way
was a sure loser.


Roger is a great assistant coach...but considering what must be bad
blood between Nielson and Messier, it would be a mistake to bring
him back even in that role.
---

Category: rec.sport.hockey
Cosine Similarity: 0.26607754991607463


We see around 1/4 similarity. While being the same topic, and sharing a common person (Messier), they talk about different issues. In this case, it's Messier's attitude towards the coach's approach.

In [87]:
didx = cos_sim[3][2][2]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[3][1][2]}')

---

Well, I could become a fan ... (-;

Seriously, this news coming since Thursday has effectively robbed the
Islanders and the Devils of any airtime on sports talk shows almost
everywhere that I've sampled ... in fact, the playoffs almost don't
exist now. )-; Ranger fans calling in to WFAN or to New York One's
midnight sports talk were in a mix of fury over this season and near-
orgasm over Keenan's hiring.  (Summarizing: Keenan is a winner and
will give the Broadway Bums 'da business' in pursuing the next Cup
chase ...)


This will be an interesting combination to watch ... Keenan has been
paid enough money to put up and shut up and just be a coach, but his
advice on any player moves will be listened to closely.  A lot of big
player moves will happen --- remember that Keenan got rid of Denis
Savard.  The country club days are over ...


If Paramount had given Smith an earlier sign of support and offered
Keenan the big money to put-up-and-shut-up back in January, the
Rangers might no

We see around 1/4 similarity. Again, this document shares overall topic with the main doc (both hockey and a mention of Messier), but they're not talking about the same issue. So it makes sense to see a somewhat low similarity.

In [88]:
didx = cos_sim[3][2][3]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[3][1][3]}')

---
The rumour was basically everywhere in Toronto based on reports
that Keenan has told both San Jose and Philadelphia that he
was no longer interested in pursuing further negotiations with
either team. 

The Ranger announcement is supposed to happen tomorrow supposedly.

The Rangers have so many veterans that they had to get a coach
with "weight" and a proven record...and whom they know Messier respects.
---

Category: rec.sport.hockey
Cosine Similarity: 0.1984294948915743


Same here. With around 1/5 similarity, both documents are hockey related and mention Messier, but tackle different issues.

In [89]:
didx = cos_sim[3][2][4]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[3][1][4]}')

---
Press conference at 1PM ...


Interestingly, Keenan's co-coach (or is it his "Number One"?) on Team
Canada at the World Championships is Roger Neilsen.  

It'd be interesting if the Rangers call in the balance of Neilsen's
contract to be Keenan's assistant ...  Roger did do a very good job
with the mediocre players, just as he handled the Cinderella Canucks
of 10 years ago ... but his mistake was playing the Rangers like those
Canucks last May ...

gld
---

Category: rec.sport.hockey
Cosine Similarity: 0.15423166072262096


Here we see an even lower similarity, which makes sense since the only things the docs have in common are hockey and a Keenan mention.

#### 5th Doc

In [90]:
didx = cos_sim[4][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')

---

The answer is obvious: ZX-11 D.
---

Category: rec.motorcycles


It's hard to say what this document is about. The topic is motorcycles, so perhaps it's talking about serial number for some specific parts or a specific motorcycle model.

In [91]:
didx = cos_sim[4][2][0]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[4][1][0]}')

---
	You know what my answer will be: Hrivnak! The choice is obvious.



---

Category: rec.sport.hockey
Cosine Similarity: 0.33460668748406375


This is an interesting one. With similarity of around 1/3, it's clear why it arises. The topic is completely different, but the overall intention of both documents is the same: providing an answer. This is why we see some similarity here.

In [92]:
didx = cos_sim[4][2][1]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[4][1][1]}')

---massacre.

Answer: a(1-1/2-1/4-1/11)=280 -> a = 1760
---

Category: talk.politics.mideast
Cosine Similarity: 0.24418769311006683


We are faced with the same situation here. Similarity of around 1/4 due to both documents providing some sort of answer, in this case a numerical calculation.

In [93]:
didx = cos_sim[4][2][2]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[4][1][2]}')

---Choose any or all of the following as an answer to the above:
 
---

Category: sci.space
Cosine Similarity: 0.1884314614369192


Somewhat similar situation here. Similarity is lower, ~1/5, due to it not providing an answer but rather requesting one, in a mutiple choice style.

In [94]:
didx = cos_sim[4][2][3]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[4][1][3]}')

---
   On my 486DX33 with the Stealth 24 VLB I get 11.4 WinMarks with ver. 3.11






   

---

Category: comp.sys.ibm.pc.hardware
Cosine Similarity: 0.18570174330961678


This one has around 1/5 similarity. It's hard to see why the similarity would be as high as it is. Docs belong to completely different categories. It could be because both have some sort of ID in them and are quite short, so the IDs make up a big porton of the overall document.

In [95]:
didx = cos_sim[4][2][4]
print(f'---{newsgroups_train.data[didx]}\n---\n')
print(f'Category: {newsgroups_train.target_names[newsgroups_train.target[didx]]}')
print(f'Cosine Similarity: {cos_sim[4][1][4]}')

---Ummm...did you have any bikes other than that KX80? If not, I'd suggest you 
look for an '89 ZX-7, since they only have about 90 horsepower, whereas the
'90 has over 100 and might be a bit much for you...

Sincerely,
Nathaniel
---

Category: rec.motorcycles
Cosine Similarity: 0.17004641138566626


It's easy to see why a document like this could be among the most similar. Both talk about motorcycles and have some sort of ID talk. Considering the other documents we've seen, it's not surprising a document like this one appeared. The fact that they share a category could very well be a coincidence, though.

### 2)

In [179]:
class ZeroShotModel:

    def __init__(self, train, vect=TfidfVectorizer()):
        self.vect = vect
        self.label = train.target_names
        self.label_idx = train.target
        self.train = self.vect.fit_transform(train.data)

    def predict(self, test_doc):
        # Get cosine simil (transform 1st)
        test = self.vect.transform([test_doc])
        cossim = cosine_similarity(test, self.train)[0]

        # Get doc idx with highest similarity.
        idx = np.argmax(cossim)

        # Get class for said doc.
        pred = self.label[self.label_idx[idx]]

        return pred

    def predict_multiple(self, test_docs):
        # Get cosine simil (transform 1st)
        test = self.vect.transform(test_docs)
        cossim = cosine_similarity(test, self.train)

        # Get docs idx with highest similarity.
        idx = np.argmax(cossim, axis=1)

        # Get class for said docs.
        pred = [self.label[self.label_idx[i]] for i in idx]

        return pred

In [183]:
# Let's see if it works.
rand_idx = random.randint(0, len(newsgroups_test.data)-1)
print(f'Test doc index: {rand_idx}')

zs = ZeroShotModel(newsgroups_train)

print(f'Class prediction: {zs.predict(newsgroups_test.data[rand_idx])}')
print(f'Actual class: {newsgroups_test.target_names[y_test[rand_idx]]}')
print(f'Document:\n{newsgroups_test.data[rand_idx]}')

Test doc index: 6712
Class prediction: soc.religion.christian
Actual class: soc.religion.christian
Document:


So do other parts of the Bible when taken literally - i.e. the Psalms
saying the Earth does not move, or the implication the Earth is flat
with four corners, etc.  The Bible was written to teach salvation, not
history or science.


What ones?  Paryers for the dead or the intercession of saints? (Which
are taught in 2 Maccabees, Sirach, and Tobit)


By your own subjective judgement.  This falling short is your judgement,
and you are not infallible - rather the Church of Jesus Christ is (see 1
Timothy 3.15).


More subjective feelings.   This is not a proof of anything more than
one persons feelings.


As I have written time and again, the Hebrew canon was fixed in Jamnia,
Palestine, in 90 AD.  60 years after the foundation of the one, holy,
catholic, and apostolic Church.  Furthermore, the opinons of Jerome do
not count.  He was neither the Church, or the Pope, or an ecumenical

It works for some documents. For others it misses. Which is expected, since it's a very simple model.

### 3)

In [184]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

In [190]:
### Let's do a small grid search here.
# Pipeline
pipeline = Pipeline([
    ('vect', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# Grid
param_grid = {
    # Vectorizer. This is an ignore of words that appear in a percentage of docs.
    'vect__max_df': [0.5, 0.75, 1.0],
    
    # Classifiers. Try both classifiers with different smoothing alphas.
    'clf': [MultinomialNB(), ComplementNB()],
    'clf__alpha': [0.01, 0.1, 0.5, 1.0]
}

# Grid search with f1 macro
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    scoring='f1_macro', 
    cv=5, 
    n_jobs=-1
)

In [191]:
# Do the search
grid_search.fit(newsgroups_train.data, newsgroups_train.target)

,estimator,Pipeline(step...inomialNB())])
,param_grid,"{'clf': [MultinomialNB(), ComplementNB()], 'clf__alpha': [0.01, 0.1, ...], 'vect__max_df': [0.5, 0.75, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [192]:
# Let's see the best model's parameters.
best_model = grid_search.best_estimator_
print(f"Parameters from grid search: {grid_search.best_params_}")

Parameters from grid search: {'clf': ComplementNB(), 'clf__alpha': 0.1, 'vect__max_df': 1.0}


The best model was obtained using ComplementNB with 0.1 smoothing alpha, without ignoring any words.

In [193]:
# Let's do a classification report for the test data.
y_pred = best_model.predict(newsgroups_test.data)
print(classification_report(newsgroups_test.target, y_pred, target_names=newsgroups_test.target_names))

                          precision    recall  f1-score   support

             alt.atheism       0.33      0.47      0.39       319
           comp.graphics       0.72      0.72      0.72       389
 comp.os.ms-windows.misc       0.71      0.55      0.62       394
comp.sys.ibm.pc.hardware       0.64      0.69      0.66       392
   comp.sys.mac.hardware       0.77      0.71      0.74       385
          comp.windows.x       0.80      0.79      0.79       395
            misc.forsale       0.74      0.72      0.73       390
               rec.autos       0.79      0.74      0.77       396
         rec.motorcycles       0.81      0.77      0.79       398
      rec.sport.baseball       0.90      0.84      0.87       397
        rec.sport.hockey       0.86      0.95      0.90       399
               sci.crypt       0.76      0.80      0.78       396
         sci.electronics       0.71      0.57      0.63       393
                 sci.med       0.76      0.79      0.77       396
         

We see the model overall performs well. Except for two categories (alt.atheism and talk.religion.misc) where it performs particularly bad across all metrics and f1 score specifically, all others get reasonable metrics over all. We also see that both the macro avg and weighted avg are similar, which is to be expected since the dataset doesn't show an extreme imbalance or anything like that.

### 4)

In [197]:
# First, let's get the document-term matrix. We transform the train data again.
dt_matrix = best_model.named_steps['vect'].transform(newsgroups_train.data)

# Get the transpose.
td_matrix = dt_matrix.T

In [198]:
type(td_matrix)

scipy.sparse._csc.csc_matrix

In [220]:
# Find 5 words from the corpus. Make sure they're `interpretable`.
# We'll select words that are highly related to categories, to make sure
# they're high frequency.
# Words are:
words = ['bible', 'circuit', 'doctor', 'server', 'wheel']

In [221]:
# Now, we should get the vectors that represent these words.
# How to do this? Mask the term-doc matrix with a bool array where our words match the feature names.
word_vectors = [
    td_matrix[
        np.where(best_model.named_steps["vect"].get_feature_names_out() == word)[0][0]
    ]
    for word in words
]

In [222]:
# Now calculate cosine similarity for each on the whole matrix.
cos_sim = [cosine_similarity(word_vector, td_matrix)[0] for word_vector in word_vectors]

In [223]:
# Get the top 5 words in similarity (ignore first since it's the word itself)
word_idx = [np.flip(cosim.argsort())[1:6] for cosim in cos_sim]

In [224]:
# Print words and the 5 most similars to it.
for idx, (word, arr) in enumerate(zip(words, word_idx)):
    print('----')
    print(word)
    print('Top 5 similar words:')
    for widx in arr:
        print(f'Word: {best_model.named_steps["vect"].get_feature_names_out()[widx]}. Cosine Similarity: {cos_sim[idx][widx]}')

----
bible
Top 5 similar words:
Word: god. Cosine Similarity: 0.26161600372794847
Word: christians. Cosine Similarity: 0.24730245175700374
Word: contadictions. Cosine Similarity: 0.21886879151712463
Word: necessitate. Cosine Similarity: 0.20873036003693882
Word: l0c. Cosine Similarity: 0.1981082753919784
----
circuit
Top 5 similar words:
Word: microstrip. Cosine Similarity: 0.2734793880395432
Word: 3ghz. Cosine Similarity: 0.2734793880395432
Word: droolproof. Cosine Similarity: 0.25840401923813966
Word: demultiplexer. Cosine Similarity: 0.24875209011626778
Word: momentarily. Cosine Similarity: 0.24214223852001157
----
doctor
Top 5 similar words:
Word: receptionist. Cosine Similarity: 0.43922432865598793
Word: clinic. Cosine Similarity: 0.327126983853701
Word: misbehavior. Cosine Similarity: 0.29510453717547797
Word: urgent. Cosine Similarity: 0.29468506875609823
Word: angering. Cosine Similarity: 0.2943499913518327
----
server
Top 5 similar words:
Word: client. Cosine Similarity: 0.283

First word chosen was `bible`, since there are a few religious categories. In the top 5 similar words we encounter both `god` and `christians`, which make sense over all, since the bible is specific to christians and god is often the word use to refer to the christian god. As for the other three, it's hard to say. First, we see `contadictions`, which might be a misspelling of `contradictions`. The bible itself might have some contradictions in its text, since, as far as I understand, it's a collection of writings by multiple people who offer varying accounts on things. Then we have `necessitate`. I'm not sure why this word has high similarity with `bible`, and I'd rather not speculate given the meaning of this word. Finally, we have `l0c`, which I'm not sure what it even means. An internet search suggest it might be a result of data corruption or a conversion artifact, but I'm not sure.

Second word chosen was `circuit`, with the hope to see both electronic terms due to the electronics and hardware categories, but also see racing terms due to the autos and motorcycles categories. We do see three terms associated with electronics: `microstrip`, which is a transmission line on a circuit board, and `3ghz`, meanining 3 GHz, a frequency, which is often among the specifications of an electric circuit, and `demultiplexer`, which is a logic circuit. All of these make sense to have here. The word `droolproof` I'm not sure on the meaning. As far as I know, it means "idiot proof", which is often applied to certain objects where things can go wrong and they are designed so people won't accidentally break or mess with them. I suppose this could be related to electrical circuits, since security against shorts and other phenomena is common. Lastly, we have `momentarily`, which I'm not exactly sure why it's here.

Third word was `doctor`, with the hope to see not only medical but also scientific terms due to the large number of scientific/computing categories. We see three words that seem related to the medical field: `receptionist`, `clinic`, and `urgent`. All of these make sense, with `urgent` being usually seen in medicine contexts in things like urgent care. Both `misbehavior` and `angering` I'm not sure why are here. I suppose since these are news articles, medical misbehavior might be a common topic in these.

Fourth was `server`, due to the computing categories. We see both `client` and `clients`, which are terms for whoever is communicating with the server. The word `shm` I had to research. Apparently it's short for shared memory, a protocol for accessing common data in memory by multiple processes. This might be related to servers in that they are required to handle multiple clients simultaneously, so they must be equiped with these kind of mechanisms for handling data. `openwindows` and `desqview` are both old graphic environments. Not sure why they're here, but it could be due to the fact that `server` is a term also used when talking about a display server, which is a key component in the GUI of any operating system.

Finally, we chose `wheel`, due to some vehicle related categories. The top word was `steering`, since a steering wheel is the wheel used to direct a vehicle. Second is `hops`, probably signaling `wheel hops`, which are a very common thing both on motorcycles and bikes, but also on cars. Then we have `_rear_`, which probably means `rear-wheel`, which is a configuration in which the engine sends power only to the rear wheels. For `flink` I suppose it makes sense for it to be here, since it means "agile", which is something one would desire their vehicles to be. As for `belasco`, I'm not sure why it's here. I wasn't able to find anything on `belasco`, a word with which I wasn't familiar, so it's hard to say anything on it. 